# Exploratory Data Analysis — Decanter Wine Dataset

Initial exploration of ~10,000 wine reviews scraped from Decanter. Goals:
- Understand the shape and quality of the data
- Identify null values and decide how to handle them
- Analyse score, colour, country, and region distributions
- Check description lengths (key field for embeddings)
- Detect and inspect duplicate entries

## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## Load Data

In [ ]:
df = pd.read_csv("../data/raw/wine_dataset.csv")
df.head()

In [ ]:
df.info()

## Null Values

Check which fields have missing data. Fields like `Brand` and `Appellation` are expected to be sparse — not every wine has a sub-label or MGA. `description` nulls are critical: wines without tasting notes cannot be embedded and will be dropped in preprocessing.

In [ ]:
nulls = df.isnull().sum()
nulls_pct = (nulls / len(df) * 100).round(1)

null_df = pd.DataFrame({
    "null_count": nulls,
    "null_pct": nulls_pct
}).sort_values("null_pct", ascending=False)

null_df[null_df["null_count"] > 0]

## Colour Distribution

The dataset is dominated by red and white wines. This imbalance matters for fine-tuning — we stratify train/val splits by `Colour × Body` to avoid the model seeing disproportionate colour groups.

In [ ]:
df["Colour"].value_counts().plot(kind="bar", figsize=(8, 4), title="Wines by Colour")
plt.xticks(rotation=0)
plt.ylabel("Count")
plt.tight_layout()
plt.show()

In [ ]:
df["Colour"].value_counts()

## Country & Region Distribution

France, USA, and Italy account for the majority of reviews. This reflects Decanter's editorial focus rather than global wine production. The long tail of smaller countries is still useful for semantic search — rare appellations can still be matched by flavour profile.

In [ ]:
df["Country"].value_counts().head(15).plot(
    kind="bar", figsize=(10, 4), title="Top 15 Countries"
)
plt.xticks(rotation=45, ha="right")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

In [ ]:
df["Country"].value_counts().head(10)

In [ ]:
df["Region"].value_counts().head(15).plot(
    kind="bar", figsize=(10, 4), title="Top 15 Regions"
)
plt.xticks(rotation=45, ha="right")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

## Score Distribution

Decanter scores cluster between 88–95 — typical for a publication that selects wines worth reviewing. Scores below 50 are data errors (likely scraper artefacts) and are dropped in preprocessing.

In [ ]:
df["score"].dropna().plot(
    kind="hist", bins=20, figsize=(8, 4), title="Score Distribution"
)
plt.xlabel("Score")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

In [ ]:
df["score"].value_counts().head(20)

In [ ]:
# Scores below 50 are scraper artefacts — dropped in preprocessing
df[df["score"] < 50][["title", "score", "description"]]

## Description Length

`description` is the only field used for embeddings. We check length distribution to set a minimum threshold — very short descriptions (<50 chars) don't carry enough semantic signal and are filtered out in preprocessing.

In [ ]:
df["desc_length"] = df["description"].dropna().apply(len)

df["desc_length"].describe()

In [ ]:
print("=== Shortest ===")
print(df.nsmallest(3, "desc_length")[["title", "desc_length", "description"]].to_string())

print("\n=== Longest ===")
print(df.nlargest(3, "desc_length")[["title", "desc_length", "description"]].to_string())

In [ ]:
pd.set_option("display.max_colwidth", None)
df.nlargest(3, "desc_length")[["title", "desc_length", "url"]]

## Grape Varieties

Cabernet Sauvignon, Chardonnay, and Pinot Noir dominate — reflecting both Decanter's coverage and global planting patterns. Blends (multiple varieties) are treated as unknown in grape-based evaluation metrics.

In [ ]:
df["Grapes"].value_counts().head(15).plot(
    kind="bar", figsize=(10, 4), title="Top 15 Grapes"
)
plt.xticks(rotation=45, ha="right")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

## Duplicates

Some wines appear multiple times — same description, different titles. This happens when Decanter publishes the same tasting note under both a producer name and a brand name. These are deduplicated in preprocessing by description hash.

In [ ]:
print("Duplicate URLs:", df["url"].duplicated().sum())
print("Duplicate titles:", df["title"].duplicated().sum())
print("Duplicate descriptions:", df["description"].duplicated().sum())

In [ ]:
mask = df["description"].duplicated(keep=False)
df[mask].sort_values("description")[["title", "description"]].head(10)